# 11.1 - Sensitivity Analysis (High-Visibility Threshold)

Visibility-threshold sensitivity for the paper's Table 2. **This notebook depends on
notebook 11**: it reuses notebook 11's *exact* `compute_exceedence_event` (all clean
crossings) and `compute_churn_floors` (per-AS non-exceedance churn baseline), and simply
sweeps the high-visibility key over `visibility_80 / 90 / 95 / 100`.

For each threshold and address family we report:
- **crossings**: clean limit crossings for ASes that have a trustworthy churn floor
  (>= 20 non-exceedance control windows);
- **with peer loss**: crossings whose coincident absolute peer drop is > 0;
- **impactful (95th-pct floor)**: crossings whose coincident drop exceeds the AS's own
  95th-percentile non-exceedance churn floor (the paper's `impactful` definition).

The churn floor is recomputed *per threshold* (the below-limit control set shifts with the
announced count). **Cross-check:** the 95% row must reproduce notebook 11's headline
counts (IPv4 4314 crossings -> 303 impactful across 219 ASes; IPv6 1501 -> 132 across 81).

Activate the venv first.
The ASN sweep is parallelised with `multiprocessing.Pool` (aggregation kept separate).

Output: `data/processed/numbers/11.1-Sensitivity_Analysis.md`.


## Imports

In [ ]:
import os
import json
import pickle
import datetime
import pandas as pd
import numpy as np
from multiprocessing import Pool
from tqdm import tqdm


## Configuration

In [ ]:
REPO_ROOT = os.path.abspath("..")
fd = open(os.path.join(REPO_ROOT, "settings.json"))
parameters = json.load(fd)
for _k in ("DATA_DIR", "DATA_RAW_DIR", "IMAGE_DIR", "WORKING_DIR", "VISIBILITY_OUTPUT_DIR", "VISIBILITY_ANNOUNCED_OUTPUT_DIR"):
    if isinstance(parameters.get(_k), str) and not os.path.isabs(parameters[_k]):
        parameters[_k] = os.path.normpath(os.path.join(REPO_ROOT, parameters[_k]))
fd.close()

data_dir = parameters["DATA_DIR"]
start_date = parameters["START_DATE"]
end_date = parameters["END_DATE"]
tier1 = parameters["TIER1"]
tier1_asns = [item["asn"] for item in tier1]

# low -> high, matching the paper table order
THRESHOLDS = ["visibility_80", "visibility_90", "visibility_95", "visibility_100"]
threshold_labels = {
    "visibility_80": "80%",
    "visibility_90": "90%",
    "visibility_95": "95%",
    "visibility_100": "100%",
}
delta = datetime.timedelta(hours=8)
N_PROCESSES = parameters.get("N_PROCESSES", 8)
print(f"Thresholds: {THRESHOLDS} | N_PROCESSES={N_PROCESSES}")


## Open stats output file (overwrites on every run)

In [ ]:
numbers_dir = f"{data_dir}/processed/numbers"
os.makedirs(numbers_dir, exist_ok=True)
_stats = open(f"{numbers_dir}/11.1-Sensitivity_Analysis.md", "w")
_stats.write("# Stats: 11.1-Sensitivity_Analysis\n\n")
_stats.write(f"*Generated: {datetime.datetime.now().strftime('%Y-%m-%d %H:%M')}*\n\n")
_stats.write("Impactful = coincident peer drop exceeds the AS's own 95th-pct "
             "non-exceedance churn floor (per notebook 11). Sweep recomputes the "
             "floor per visibility threshold.\n\n")
print("Stats file opened.")


## Load data

Same inputs as notebook 11 (PeeringDB limits, announced-prefix timeseries, peer timeseries, selected ASNs).

In [ ]:
filename = (
    f"{data_dir}/processed/peeringdb/prefix_limit_peeringdb_{start_date}_{end_date}.pkl"
)
df_peeringdb = pd.read_pickle(filename)

filename = f"{data_dir}/processed/timeseries_prefix_announced_visibility.pkl"
with open(filename, "rb") as fd:
    announced_prefixes = pickle.load(fd)

filename = f"{data_dir}/processed/peers/df_peers_ipv4.pkl"
with open(filename, "rb") as f:
    peers_ipv4 = pickle.load(f)
filename = f"{data_dir}/processed/peers/df_peers_ipv6.pkl"
with open(filename, "rb") as f:
    peers_ipv6 = pickle.load(f)

with open(f"{data_dir}/processed/selected_asns.pkl", "rb") as fd:
    selected_asns = pickle.load(fd)

# restrict to the selected ASNs (same as notebook 11)
announced_prefixes = {
    asn: announced_prefixes[asn] for asn in selected_asns if asn in announced_prefixes
}
df_peeringdb = df_peeringdb[df_peeringdb["asn"].isin(selected_asns)].copy()
print(f"selected ASNs: {len(selected_asns)} | with announced prefixes: {len(announced_prefixes)}")


## Crossing detection and per-AS churn floor

**Ported verbatim from notebook 11** (cells `compute_exceedence_event` and `compute_churn_floors`). Do not edit here; keep in sync with notebook 11.

In [ ]:
def compute_exceedence_event(asn, ipv, high_visibility="visibility_95"):
    """Return ALL clean limit crossings for (asn, ipv).

    A crossing = below the limit at t-2 and t-1, then above at t, with no
    temporal gap. We NO LONGER require a peer drop here (that filtering used to
    bake in the arbitrary 1% rule and the <5-peers guard). Every crossing is
    returned with its coincident ABSOLUTE peer drop, computed with a
    one-window-ahead horizon (enforcement can land the snapshot after the
    crossing). Whether a crossing is "impactful" is decided later, against the
    AS's own per-AS churn floor (see compute_churn_floors).
    """

    default_output = []

    # if there is no announced prefix for this ASN and IP version, we can ignore it
    if asn not in announced_prefixes:
        return default_output
    if ipv not in announced_prefixes[asn]:
        return default_output

    announced_prefixes_asn_ipv = announced_prefixes[asn][ipv][high_visibility]
    if len(announced_prefixes_asn_ipv) == 0:
        return default_output

    announced_prefixes_asn_ipv = {
        k: v for k, v in sorted(announced_prefixes_asn_ipv.items(), key=lambda x: x[0])
    }

    peers_ipv = peers_ipv4 if ipv == 4 else peers_ipv6
    peers_ipv_asn = peers_ipv[peers_ipv["asn"] == asn]
    if peers_ipv_asn.empty:
        print(f"ASN {asn} has no peer data for IP version {ipv}")
        return default_output

    peers_ipv_asn = peers_ipv_asn.iloc[0]
    peers_ipv_asn_dates = peers_ipv_asn["datetime"]
    peers_ipv_asn_num_peers = peers_ipv_asn["num_peers"]
    peers_ipv_asn = {
        date: num_peers
        for date, num_peers in zip(peers_ipv_asn_dates, peers_ipv_asn_num_peers)
    }
    peers_ipv_asn = {
        date: num_peers
        for date, num_peers in sorted(peers_ipv_asn.items(), key=lambda x: x[0])
    }

    # by default asn should exist in peeringdb since we are using selected_asns
    prefix_limits_asn = df_peeringdb[df_peeringdb["asn"] == asn].iloc[0]
    # however, it may be that there is no limit for this ASN and IP version
    prefix_limits_asn_ipv_date = prefix_limits_asn["dates"]
    prefix_limits_asn_ipv_count = prefix_limits_asn[f"limits_ipv{ipv}"]
    if prefix_limits_asn_ipv_count is None or prefix_limits_asn_ipv_count == 0:
        return default_output

    prefix_limits_asn_ipv = {
        date: count
        for date, count in zip(prefix_limits_asn_ipv_date, prefix_limits_asn_ipv_count)
    }

    # population guardrails: below the limit sometimes, above sometimes, and
    # below for at least half the year, so it has a meaningful non-exceedance
    # baseline to build a per-AS churn floor from.
    count_below = 0
    for date in announced_prefixes_asn_ipv:
        if date in prefix_limits_asn_ipv:
            if announced_prefixes_asn_ipv[date] <= prefix_limits_asn_ipv[date]:
                count_below += 1

    # discard if announce is always below limit
    if count_below == 0:
        return default_output

    # discard if announce is always above limit
    if count_below == len(announced_prefixes_asn_ipv):
        return default_output

    # discard if announce is below limit for less than half of the time
    if count_below < len(announced_prefixes_asn_ipv) / 2:
        return default_output

    # kept only for the plotting helpers below
    large_number_peers = np.mean(list(peers_ipv_asn.values())) > 10

    announced_prefixes_asn_ipv_dates = list(announced_prefixes_asn_ipv.keys())

    excedence_events = []
    for index in range(2, len(announced_prefixes_asn_ipv) - 1):

        previous_2_date = announced_prefixes_asn_ipv_dates[index - 2]
        previous_date = announced_prefixes_asn_ipv_dates[index - 1]
        current_date = announced_prefixes_asn_ipv_dates[index]
        next_date = announced_prefixes_asn_ipv_dates[index + 1]

        # check that there is no temporal gap between the dates
        if not (
            next_date - delta == current_date
            and current_date - delta == previous_date
            and previous_date - delta == previous_2_date
        ):
            continue

        # structural crossing: below at t-2 and t-1, then above at t
        if not (
            announced_prefixes_asn_ipv[previous_2_date]
            <= prefix_limits_asn_ipv[previous_2_date]
            and announced_prefixes_asn_ipv[previous_date]
            <= prefix_limits_asn_ipv[previous_date]
            and announced_prefixes_asn_ipv[current_date]
            > prefix_limits_asn_ipv[current_date]
        ):
            continue

        n_previous_peers = peers_ipv_asn[previous_date]
        n_current_peers = peers_ipv_asn[current_date]
        n_next_peers = peers_ipv_asn[next_date]

        n_previous_prefixes = announced_prefixes_asn_ipv[previous_date]
        n_current_prefixes = announced_prefixes_asn_ipv[current_date]

        # coincident ABSOLUTE peer drop, one-window-ahead horizon.
        # positive = peers lost. deepest peer count over {t, t+1}.
        deepest = min(n_current_peers, n_next_peers)
        abs_drop = n_previous_peers - deepest
        # is the deeper drop reflected at t+1 rather than t? drives drop_date in
        # the detailed analysis (matches the old check_next semantics)
        check_next = n_next_peers < n_current_peers
        percentage_drop = (
            abs_drop / n_previous_peers * 100 if n_previous_peers > 0 else 0.0
        )

        excedence_event = {
            "asn": asn,
            "ipv": ipv,
            "date": current_date,
            "peeringdb_limit_previous_date": prefix_limits_asn_ipv[previous_date],
            "peeringdb_limit_date": prefix_limits_asn_ipv[current_date],
            "n_prefixes_previous_date": n_previous_prefixes,
            "n_prefixes_date": n_current_prefixes,
            "n_previous_peers": n_previous_peers,
            "n_current_peers": n_current_peers,
            "n_next_peers": n_next_peers,
            "check_next": check_next,
            "abs_drop": abs_drop,
            "percentage_drop": percentage_drop,
            "is_visually_interesting": bool(
                percentage_drop > 10 and large_number_peers
            ),
        }
        excedence_events.append(excedence_event)

    return excedence_events


In [ ]:
# Per-AS "normal churn" baseline (replaces the arbitrary 1% peer-drop rule).
#
# For every AS that crosses its limit, we measure its typical peer loss from the
# windows in which it operates BELOW its limit (non-exceedance), and read off the
# 90/95/99th percentiles. A crossing is later called "impactful" only if its
# coincident peer drop exceeds that AS's OWN 95th-pct floor -- the bar is per-AS
# (size-aware) and data-driven, not a fixed 1%.
#
# Metric: ABSOLUTE peer drop, one-window-ahead horizon, computed identically to
# the crossings. For the control we only look ahead INTO another below-limit
# window, so a crossing can never leak into its own baseline (no arbitrary
# buffer needed -- the baseline lives entirely below the limit).

CHURN_PERCENTILES = (90, 95, 99)
MIN_CONTROL_WINDOWS = 20  # need >= this many non-exceedance transitions for a floor


def compute_churn_floors(asn, ipv, high_visibility="visibility_95"):
    """Return (floors_abs, control_abs, control_rel) or None if too few windows.

    floors_abs  : {90: x, 95: y, 99: z} absolute-peer-drop percentiles
    control_abs : list of absolute drops over below-limit windows
    control_rel : same drops as a % of previous peers (for the pooled CDF)
    """
    if asn not in announced_prefixes or ipv not in announced_prefixes[asn]:
        return None
    announced = announced_prefixes[asn][ipv][high_visibility]
    if len(announced) == 0:
        return None
    announced = {k: v for k, v in sorted(announced.items(), key=lambda x: x[0])}

    peers_ipv = peers_ipv4 if ipv == 4 else peers_ipv6
    peers_ipv_asn = peers_ipv[peers_ipv["asn"] == asn]
    if peers_ipv_asn.empty:
        return None
    peers_ipv_asn = peers_ipv_asn.iloc[0]
    peers = {
        date: num_peers
        for date, num_peers in zip(
            peers_ipv_asn["datetime"], peers_ipv_asn["num_peers"]
        )
    }

    prefix_limits_asn = df_peeringdb[df_peeringdb["asn"] == asn].iloc[0]
    limits = prefix_limits_asn[f"limits_ipv{ipv}"]
    if limits is None:
        return None
    limits = {date: c for date, c in zip(prefix_limits_asn["dates"], limits)}

    dates = list(announced.keys())

    def below(d):
        return d in limits and announced[d] <= limits[d]

    control_abs = []
    control_rel = []
    for i in range(1, len(dates)):
        prev_d, cur_d = dates[i - 1], dates[i]
        # consecutive, both endpoints below the limit (non-exceedance churn)
        if cur_d - delta != prev_d:
            continue
        if not (below(prev_d) and below(cur_d)):
            continue
        if prev_d not in peers or cur_d not in peers:
            continue
        pb = peers[prev_d]
        if pb == 0:
            continue

        # one-window-ahead horizon, but only INTO another below-limit window
        cand = [peers[cur_d]]
        if i + 1 < len(dates):
            nxt_d = dates[i + 1]
            if nxt_d - delta == cur_d and below(nxt_d) and nxt_d in peers:
                cand.append(peers[nxt_d])
        drop = pb - min(cand)

        control_abs.append(drop)
        control_rel.append(drop / pb * 100)

    if len(control_abs) < MIN_CONTROL_WINDOWS:
        return None

    floors_abs = {q: float(np.percentile(control_abs, q)) for q in CHURN_PERCENTILES}
    return floors_abs, control_abs, control_rel


## Per-ASN worker (Pool unit of work)

One ASN's counts for a given (family, threshold). Relies on the module globals inherited by `fork`; keep the two functions above verbatim from notebook 11.

In [ ]:
def eval_asn(args):
    """Return per-ASN counts for one (asn, ipv, threshold):
    (crossings, any_loss, imp90, imp95, imp99, no_floor, has_imp95).
    crossings are counted only for ASes that have a trustworthy floor,
    matching notebook 11's classify loop."""
    asn, ipv, threshold = args
    crossings = compute_exceedence_event(asn, ipv, high_visibility=threshold)
    if not crossings:
        return (0, 0, 0, 0, 0, 0, 0)
    floors_res = compute_churn_floors(asn, ipv, high_visibility=threshold)
    if floors_res is None:
        return (0, 0, 0, 0, 0, 1, 0)  # crossing(s) exist but no trustworthy floor
    floors_abs, _control_abs, _control_rel = floors_res
    n_cross = len(crossings)
    n_any = i90 = i95 = i99 = 0
    for ev in crossings:
        d = ev["abs_drop"]
        if d > 0:
            n_any += 1
            if d > floors_abs[90]:
                i90 += 1
            if d > floors_abs[95]:
                i95 += 1
            if d > floors_abs[99]:
                i99 += 1
    has95 = 1 if i95 > 0 else 0
    return (n_cross, n_any, i90, i95, i99, 0, has95)


## Run the visibility-threshold sweep (Pool over ASNs)

For each threshold and family, map `eval_asn` over the selected ASNs and aggregate. The floor is recomputed at each threshold.

In [ ]:
results = {}
for threshold in THRESHOLDS:
    print(f"\n=== {threshold_labels[threshold]} ({threshold}) ===")
    res = {}
    for ipv in [4, 6]:
        args = [(asn, ipv, threshold) for asn in selected_asns]
        agg = [0] * 7
        with Pool(N_PROCESSES) as pool:
            for r in tqdm(pool.imap_unordered(eval_asn, args, chunksize=50),
                          total=len(args), desc=f"  IPv{ipv}", leave=False):
                for k in range(7):
                    agg[k] += r[k]
        res[ipv] = {
            "crossings": agg[0], "any_loss": agg[1],
            "impactful_90": agg[2], "impactful_95": agg[3], "impactful_99": agg[4],
            "no_floor": agg[5], "ases_95": agg[6],
        }
        print(f"  IPv{ipv}: {agg[0]} crossings | {agg[1]} w/ peer loss | "
              f"impactful@95={agg[3]} (across {agg[6]} ASes) [@90={agg[2]}, @99={agg[4]}]")
    results[threshold] = res
print("\nDone.")


## Cross-check against notebook 11 (95% row)

In [ ]:
expected = {4: {"crossings": 4314, "impactful_95": 303, "ases_95": 219},
            6: {"crossings": 1501, "impactful_95": 132, "ases_95": 81}}
got = results["visibility_95"]
_stats.write("## Cross-check vs notebook 11 (95% row)\n\n")
_stats.write("| IP | crossings (got/exp) | impactful@95 (got/exp) | ASes (got/exp) | match |\n")
_stats.write("|----|--------------------|------------------------|----------------|-------|\n")
for ipv in [4, 6]:
    g, e = got[ipv], expected[ipv]
    ok = all(g[k] == e[k] for k in e)
    _stats.write(f"| IPv{ipv} | {g['crossings']}/{e['crossings']} | "
                 f"{g['impactful_95']}/{e['impactful_95']} | {g['ases_95']}/{e['ases_95']} | "
                 f"{'YES' if ok else 'NO -- investigate'} |\n")
    print(f"IPv{ipv}: crossings {g['crossings']}/{e['crossings']}, "
          f"impactful@95 {g['impactful_95']}/{e['impactful_95']}, "
          f"ASes {g['ases_95']}/{e['ases_95']} -> {'OK' if ok else 'MISMATCH'}")
_stats.write("\n")


## Write the sensitivity table (source for paper Table 2)

In [ ]:
_stats.write("## Sensitivity Table (visibility threshold vs. impactful)\n\n")
_stats.write("Paper Table 2 uses the **crossings** and **impactful (95th-pct floor)** columns. "
             "`with peer loss` (any positive drop) is reported for reference; `@90`/`@99` show "
             "robustness to the floor percentile at each threshold.\n\n")
_stats.write("| Threshold | IPv4 crossings | IPv4 w/ peer loss | IPv4 impactful@95 | "
             "IPv6 crossings | IPv6 w/ peer loss | IPv6 impactful@95 |\n")
_stats.write("|-----------|---------------|-------------------|-------------------|"
             "---------------|-------------------|-------------------|\n")
for t in THRESHOLDS:
    r4, r6 = results[t][4], results[t][6]
    star = " (baseline)" if t == "visibility_95" else ""
    _stats.write(f"| {threshold_labels[t]}{star} | {r4['crossings']} | {r4['any_loss']} | "
                 f"{r4['impactful_95']} | {r6['crossings']} | {r6['any_loss']} | "
                 f"{r6['impactful_95']} |\n")
_stats.write("\n")

# secondary: floor-percentile robustness at each threshold
_stats.write("### Impactful under 90/95/99th per-AS floor (per threshold)\n\n")
_stats.write("| Threshold | IPv4 @90 | IPv4 @95 | IPv4 @99 | IPv6 @90 | IPv6 @95 | IPv6 @99 |\n")
_stats.write("|-----------|----------|----------|----------|----------|----------|----------|\n")
for t in THRESHOLDS:
    r4, r6 = results[t][4], results[t][6]
    _stats.write(f"| {threshold_labels[t]} | {r4['impactful_90']} | {r4['impactful_95']} | "
                 f"{r4['impactful_99']} | {r6['impactful_90']} | {r6['impactful_95']} | "
                 f"{r6['impactful_99']} |\n")
_stats.write("\n")
print("Sensitivity tables written.")


## LaTeX snippet for paper Table 2 (crossings + impactful@95)

In [ ]:
print("% ---- paste into tab:sensitivity-threshold (crossings | impactful@95) ----")
print(r"\begin{tabular}{@{}lcccc@{}}")
print(r"\toprule")
print(r"\textbf{Threshold} & \multicolumn{2}{c}{\textbf{IPv4}} & \multicolumn{2}{c}{\textbf{IPv6}} \\")
print(r"\cmidrule(lr){2-3}\cmidrule(lr){4-5}")
print(r" & \textbf{Crossings} & \textbf{Impactful} & \textbf{Crossings} & \textbf{Impactful} \\")
print(r"\midrule")
for t in THRESHOLDS:
    r4, r6 = results[t][4], results[t][6]
    row = f"{threshold_labels[t]} & {r4['crossings']} & {r4['impactful_95']} & {r6['crossings']} & {r6['impactful_95']} \\\\"
    if t == "visibility_95":
        row = r"\rowcolor{green!15}" + row
    print(row)
print(r"\bottomrule")
print(r"\end{tabular}")


In [ ]:
_stats.close()
print(f"Stats written to {numbers_dir}/11.1-Sensitivity_Analysis.md")
